# W1 — Data loading, cleaning and feature engineering
**Owner:** Xuan · **Reviewer:** Andrea

_Section text budget: ~600 words. Log any AI use on the Trello AI usage log card._

## 1. Data source
- UN General Debate Corpus (UNGDC), 1946–2025. 
    - Source: https://www.ungdc.bham.ac.uk/ 
    - Download UNGDC_1946-2025.tar.gz and place it in data/raw/ (not committed: 71 MB, too large)
- UCDP/PRIO Armed Conflict Dataset v26.1. 
    - Source: https://ucdp.uu.se/downloads/ (Included in data/raw/)

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42  # shared seed for reproducibility

In [ ]:
import tarfile
from pathlib import Path

RAW = Path('../data/raw')


rows = []
# read file straight from the archive -> no unzipping, keeps the data raw
with tarfile.open(RAW / 'UNGDC_1946-2025.tar.gz') as tar:
    for member in tar:
        name = Path(member.name).name
        parts = name.removesuffix('.txt').split('_')
        if not name.endswith('.txt') or name.startswith('._') or len(parts) != 3:
            continue
        iso3, session, year = parts
        text = tar.extractfile(member).read().decode('utf-8')
        rows.append({'iso3': iso3, 'session': int(session), 'year': int(year), 'text': text})

In [ ]:
# One row per speech
speeches_raw = pd.DataFrame(rows)
print(speeches_raw.shape)
speeches_raw.head()


In [ ]:
# One row per conflict per year
conflict_raw = pd.read_csv(RAW / 'UcdpPrioConflict_v26_1.csv')
print(conflict_raw.shape)
conflict_raw.head()

## 2. First look
- Speeches: 11,141 speeches, 200 country codes, 1946-2025 (39 speeches in 1946, ~190 a year by the 2020s as the UN grew). No missing or duplicate values in country-years. 
    - Length: 2,362-127,150 characters.
    - Text issues: 2,935 speeches start with a stray encoding character and 3,966 contain paragraph numbers. Removed in section 3.
- Conflicts: 2,816 conflict-years, 1946-2025, no duplicates. Missing values are structural, not errors (e.g. gwno_b is empty when side B is a rebel group), so they ar enot filled. 
Country codes are stored as text because some cells list several countries (e.g. "645, 2, 200"). Split in section 3
    - Country codes: 4 historical states (Czechoslovakia, East Germany, South Yemen and Yugoslavia) and the EU appear, in addition to the Holy Sea and Palestine. These need handling. 
- Missing speeches: within 1946-2024, 137 countries skip at least one year. These have no speech to predict from. 
- Multi-country cells: 5 'gwno_a' and 'gwno_b' cells list several countries, these are split in the next section. 

Note: country-year refers to one speech per one country, the conflict data is converted to the same unit so both conflict and speech can be matched together. 


### 2.1 Speech data

In [ ]:
# Preview the structure of the speeches data
speeches_raw.head()

In [ ]:
# How many speeches per year? 
speeches_raw['year'].value_counts().sort_index()

In [ ]:
# What are the speeches length? -> check for empty or truncated files
speeches_raw['text'].str.len().describe()

In [ ]:
# What are the shortest and longest speeches? 
shortest = speeches_raw.loc[speeches_raw['text'].str.len().idxmin()]
longest = speeches_raw.loc[speeches_raw['text'].str.len().idxmax()]
print(shortest[['iso3', 'year']], shortest['text'][:500])
print(longest[['iso3', 'year']], longest['text'][:500])

In [ ]:
# Duplicate speeches check, does any country have two speeches in the same year? 
speeches_raw.duplicated(['iso3', 'year']).sum()

In [ ]:
# How many speeches a country has had over the years? 
speeches_raw['iso3'].value_counts().sort_values(ascending=False)

In [ ]:
# How many country codes are there?
speeches_raw['iso3'].nunique()

In [ ]:
# Is there missing data in the speech data set? 
speeches_raw.isna().sum()

In [ ]:
# How many speeches contain text problems? (stray start character, paragraph numbers)
print(speeches_raw['text'].str.startswith('\ufeff').sum())
print(speeches_raw['text'].str.contains(r'^\d+\.\s', regex=True, flags=re.M).sum())

In [ ]:
# Which codes stopped appearing before 2020? e.g. states that do not exist anymore, or are not standard 3 letter code. 
codes = speeches_raw.groupby('iso3')['year'].agg(['min', 'max', 'count'])
codes[(codes['max'] < 2020) | (codes.index.str.len() != 3)]

In [ ]:
# Do countries in the data set skip years? Which country-years have no speech?
in_range = speeches_raw[speeches_raw['year'].between(1946, 2024)]
span = in_range.groupby('iso3')['year'].agg(['min', 'max', 'count'])
span['missing_years'] = span['max'] - span['min'] + 1 - span['count']
print((span['missing_years'] > 0).sum(), 'countries with missing years', span['missing_years'].sum(), 'missing country-years')
span.sort_values('missing_years', ascending=False).head()

### 2.2 Conflict data

In [ ]:
# Preview the structure of the conflict data
conflict_raw.head()

In [ ]:
# Is there any conflict code cells that list several countries? 
for col in ['gwno_a', 'gwno_b', 'gwno_loc']:
    print(col, conflict_raw[col].astype(str).str.contains(',').sum())

In [ ]:
# What type is each conflict column? 
conflict_raw.info()

In [ ]:
# Are there duplicates in the conflict data set? 
conflict_raw.duplicated(['conflict_id', 'year']).sum()

In [ ]:
# Why are the country codes stored as text? (multi-country cells like "645, 2, 200")
conflict_raw[['gwno_a', 'gwno_b']].dtypes

## 3. Cleaning
The two data sets are cleaned and put into the same format to be joined after.

### 3.1 Clean speech text
Byte-order mark and paragraph numbers found in 2.1 are removed, they are just formatting noise and not words from the speech. Stopwords and lowercasing are left for the text features (TF-IDF), so they are decided on the training data. 

In [ ]:
paragraph_num = re.compile(r'^\s*\d+\.\s', flags=re.MULTILINE)

def clean_text(text: str) -> str:
    """remove formatting noise, leaving the words untouched"""
    #remove byte order mark
    text = text.replace('\ufeff', '')
    #remove paragraph numbers
    text = paragraph_num.sub('', text)
    text = paragraph_num.sub('', text) #second pass, since 3 numbers were left in the first round
    #remove repeated spaces and tabs
    text = re.sub(r'[ \t]+', ' ', text)
    #remove runs of blank lines
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text.strip()
speeches_cleaned = speeches_raw.copy()
speeches_cleaned['text'] = speeches_cleaned['text'].map(clean_text)

# Test check
print('BOM left:', speeches_cleaned['text'].str.contains('\ufeff').sum())
print('Paragraph numbers left:', speeches_cleaned['text'].str.contains(paragraph_num).sum())
print('Median length before -> after cleaning:', speeches_raw['text'].str.len().median(), '->', speeches_cleaned['text'].str.len().median())

### 3.2 Year limit
All speeches are kept from 1946. Year 2025 is dropped because there is no later outcome to predict. 

In [ ]:
year_min, year_max = 1946, 2024

speeches_cleaned = (speeches_cleaned[speeches_cleaned['year'].between(year_min, year_max)]
                    .drop(columns=['session'])
                    .reset_index(drop = True))
print(speeches_cleaned.shape, speeches_cleaned['year'].min(), speeches_cleaned['year'].max())

print('Pargraphs numbers left (1946-2024):', speeches_cleaned['text'].str.contains(paragraph_num).sum())

### 3.3 Reshape conflict data 
UCDP has one row per conflict per year. We need one row per country per year, only counting the main sides (gwno_a and gwno_b). Troop supporters are left out

In [ ]:
conflict_cleaned = conflict_raw[['conflict_id', 'year', 'gwno_a', 'gwno_b',  'intensity_level', 'type_of_conflict']]

# Stack side A and side B: one row per country per conflict-year
conflict_long = (conflict_cleaned
                 .melt(id_vars=['conflict_id', 'year', 'intensity_level'], value_vars=['gwno_a', 'gwno_b'],
                       var_name='side', value_name='gwno')
                .dropna(subset=['gwno']))

# Split multi-country cells into separate rows: "200, 2"
conflict_long['gwno'] = conflict_long['gwno'].astype(str).str.split(',')
conflict_long = conflict_long.explode('gwno')
conflict_long['gwno'] = conflict_long['gwno'].str.strip().astype(int)

# Collapse to one row per country-year because one country can be involved in multiple conflicts in the same year
conflict_conYear =(conflict_long
                   .groupby(['gwno', 'year'])
                   .agg(intensity_level=('intensity_level', 'max'), n_conflicts=('conflict_id', 'nunique'))
                   .reset_index())

print(conflict_conYear.shape, conflict_conYear['gwno'].nunique(), 'countries')
print('Country-years in conflict 1946-2025:', conflict_conYear['year'].between(1991, 2025).sum())


### 3.4 Translate country codes
The speech data uses letter codes (ISO3) and UCDP data uses Gleditsch-Ward numbers.  country_converter is used to build a lookup table. Where the lookup table disagrees with the UCDP coding, it is fixed manually. Non-states and and seats that did not belong to independent states are dropped. 

In [ ]:
import logging
import country_converter as coco
logging.getLogger('country_converter').setLevel(logging.ERROR)

codes = pd.DataFrame({'iso3': speeches_cleaned['iso3'].unique()})
codes['gwno'] = pd.to_numeric(
    coco.convert(codes['iso3'].tolist(), src = 'iso3', to='GWcode', not_found=None), 
    errors='coerce'   
)

# Manual fixes because the package disagress with the UCDP data set on some codes
overrides = {
    'DDR': 265, #UCDP: 265 = East Germany, package has no code (Speeches 1973-1990)
    'YEM' : 678, #UCDP codes unified Yemen as 678, package gives 680
    'VNM' : 816, #UCDP: 816 = Vietnam, packages gives 817 (South Vietnam)
    'SRB' : 345, #UCDP: 345 = Serbia, package gives 340
    'YUG' : 345, #Federal Republic of Yugoslavia (speeches up to 2005)
    'CSK' : 315, #Czechoslovakia (speeches 1991-1992)
    'CZE' : 316,  #Czech Republic, package gives 315 (Czechoslovakia)
    'YMD' : 680, #South Yemen (speeches to 1989)
}
codes['gwno'] = codes['iso3'].map(overrides).fillna(codes['gwno']).astype('Int64')

#Test checks, lookup table
print('No GW code:', codes.loc[codes['gwno'].isna(), 'iso3'].tolist())
print(codes[codes['gwno'].duplicated(keep=False) & codes['gwno'].notna()])
codes.to_csv('../data/processed/country_codes.csv', index=False )

# Attach the GW codes to the speeches data set
speeches_coded = speeches_cleaned.merge(codes, how='left', on='iso3', validate = 'many_to_one')

#China's UN seat was held by the Republic of China (Taiwan) until 1971
speeches_coded.loc[(speeches_coded['iso3'] == 'CHN') & (speeches_coded['year'] <= 1971), 'gwno'] = 713

#Drop: Ukraine/ Belarus before 1991 (UN seats as Soviet republic, not independent states) and 
# non-states with no GWO code (EU, PSE, VAT)
dropped_speeches = ((speeches_coded['iso3'].isin(['UKR', 'BLR']) & (speeches_coded['year'] < 1991)) |
                    (speeches_coded['gwno'].isna()))
print('Dropped speeches:', speeches_coded[dropped_speeches].groupby('iso3').size().to_dict())
speeches_coded = speeches_coded[~dropped_speeches].reset_index(drop=True)

unassigned = set(conflict_conYear['gwno']) - set(speeches_coded['gwno'])
#print('Conflict countries with no speeches:', unassigned) -> output: 817, 751 (South Vietnam, Hyderbadad)

# Create safety net in case the data changes these 2 codes will stand out
no_un = {
    751: 'Hyderabad (princely state, part of india since 1948)',
    817: 'South Vietnam (Republic of Vietnam 1955-1975)',
}
print('Conflict countries with no speeches:' )
for code in sorted(unassigned):
    print(f' {code}: {no_un.get(code, 'Unknown. CHECK')}')

## 4. Feature engineering
_Transformations applied and the reason for each._

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create model-ready features."""
    df = df.copy()
    # TODO
    return df

## 5. Shared train/test split
_Used by both W3 and W4._

In [ ]:
from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)